## Analyzing Factors Associated with Traffic Crash Severity in California Using 2025 CCRS Data

Students: 
- Daniel Ortega Rubio
- Jonathan Mitchell
- Sherlyn Ballestero Cruz

This project will analyze 2025 traffic crash data from the California Crash Reporting System (CCRS) to identify patterns and factors associated with crash frequency and severity. The dataset contains approximately 402,460 reported collision records from across California, with variables including date and time, location, weather and road conditions, lighting, collision type, and the number of people injured or killed. The primary **research question is whether environmental factors, such as weather and lighting, and temporal factors, such as time of day and day of the week, are significantly associated with crash severity.


 ### Step 1: load the data in to a dataframe

**Dataset:** California Crash Reporting System (CCRS), Crashes_2025 <br>
**Source:** California Open Data Portal (data.ca.gov / publish.data.ca.gov)<br>
**Direct link**: https://publish.data.ca.gov/dataset/ccrs/resource/9f4fc839-122d-4595-a146-43bc4ed16f46 <br>
**Format & Size:** CSV, approximately 186 MB, 402,460 records <br>
**Why this dataset:** Official government data, a large dataset, and a real-world traffic safety problem.<br>



In [2]:
import pandas as pd
import numpy as np

# TODO: load the data in to a dataframe
# TODO: display the head of the dataframe
url = "https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/9f4fc839-122d-4595-a146-43bc4ed16f46/download/crashes_2025.csv"
df = pd.read_csv(url, low_memory=False)

display(df.head())

print("Dataset Shape:", df.shape)

df.info()

,Collision Id,\tReport Number,\tReport Version,\tIs Preliminary,\tNCIC Code,Crash Date Time,\tCrash Time Description,\tBeat,City Id,\tCity Code,...,IsCountyRoad,IsFreeway,CHP555Version,\tIsAdditonalObjectStruck,NotificationDate,\tNotificationTimeDescription,\tHasDigitalMediaFiles,\tEvidenceNumber,\tIsLocationReferToNarrative,\tIsAOIOneSameAsLocation
0,4550265,9685-2025-00096,1,False,9685,1/18/2025 3:20:00 PM,1520,039,994.0,3300.0,...,False,False,4,NaN,1/18/2025 3:25:00 PM,1525,False,NaN,NaN,True
1,4550264,250110005,1,False,4116,1/10/2025 8:28:00 AM,828,A,1310.0,4116.0,...,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4550263,9580-2025-00214,1,False,9580,1/23/2025 8:45:00 AM,845,171,535.0,1942.0,...,False,True,4,NaN,1/23/2025 8:47:00 AM,0847,False,NaN,NaN,True
3,4550262,9765-2025-00066,1,False,9765,1/15/2025 12:30:00 PM,1230,041,1701.0,5606.0,...,False,True,4,NaN,1/15/2025 12:35:00 PM,1235,True,EMBEDDED,NaN,True
4,4550261,9426-2025-00035,1,False,9426,1/20/2025 12:30:00 PM,1230,460,397.0,1500.0,...,True,False,4,NaN,1/20/2025 12:32:00 PM,1232,False,NaN,NaN,True


Dataset Shape: (402539, 74)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 402539 entries, 0 to 402538
Data columns (total 74 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Collision Id                        402539 non-null  int64  
 1   	Report Number                      402464 non-null  object 
 2   	Report Version                     402539 non-null  int64  
 3   	Is Preliminary                     402539 non-null  bool   
 4   	NCIC Code                          402539 non-null  object 
 5   Crash Date Time                     402539 non-null  object 
 6   	Crash Time Description             402539 non-null  int64  
 7   	Beat                               381886 non-null  object 
 8   City Id                             402500 non-null  float64
 9   	City Code                          402500 non-null  float64
 10  	City Name                          402500 non-null  object 
 11

### Step2: Clean the data


In [3]:

# Use a separate DataFrame 
crash_days = df.copy()
crash_days.columns = crash_days.columns.str.strip()
crash_days = crash_days[['Collision Id', 'Crash Date Time']].copy()

# Remove exact duplicates in these selected columns.
duplicates = crash_days.duplicated().sum()
crash_days = crash_days.drop_duplicates().copy()
print('Duplicate ID/date rows removed:', duplicates)

# Convert dates and exclude records missing 
crash_days['Crash Date Time'] = pd.to_datetime(
    crash_days['Crash Date Time'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce'
)
missing_rows = crash_days.isna().any(axis=1).sum()
crash_days = crash_days.dropna().copy()
print('Rows missing an ID or valid date removed:', missing_rows)

# Select only crashes dated in 2025.
crash_days = crash_days[crash_days['Crash Date Time'].dt.year == 2025].copy()
crash_days['Day'] = crash_days['Crash Date Time'].dt.day_name()
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']




Duplicate ID/date rows removed: 0
Rows missing an ID or valid date removed: 0


In [4]:
# Count unique collision IDs for each weekday, then calculate each day's share.
weekday_summary = crash_days.groupby('Day')['Collision Id'].nunique().reindex(days, fill_value=0).to_frame('Crash count')
weekday_summary['Percent of crashes'] = (
    weekday_summary['Crash count'] / weekday_summary['Crash count'].sum() * 100
).round(2)
display(weekday_summary)
print('Total crashes counted:', weekday_summary['Crash count'].sum())
print('Day with the most reported crashes:', weekday_summary['Crash count'].idxmax())
print('Day with the fewest reported crashes:', weekday_summary['Crash count'].idxmin())

,Crash count,Percent of crashes
Day,,
Monday,55760,13.85
Tuesday,58636,14.57
Wednesday,59385,14.75
Thursday,59876,14.87
Friday,64469,16.02
Saturday,55494,13.79
Sunday,48919,12.15


Total crashes counted: 402539
Day with the most reported crashes: Friday
Day with the fewest reported crashes: Sunday
